In [1]:
# Data manipulation and preprocessing
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler

# Classification models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
import lightgbm as lgb
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np
# Evaluation metrics
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             precision_score, recall_score, f1_score, make_scorer, roc_curve, accuracy_score)

# Feature selection
from sklearn.feature_selection import SelectKBest, chi2, RFE

# Data balancing (if necessary)
from imblearn.over_sampling import SMOTE

# Handling warnings
import warnings
warnings.filterwarnings("ignore")

/usr/local/lib/python3.10/dist-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [2]:
# Load the dataset

df_filtered = pd.read_csv("combined_df.csv")

In [3]:
# Select relevant symptom columns
symptom_columns = ['Dribbling', 'Swallowing', 'Vomiting', 'Constipation', 'Bowel inconsistence',
                   'Bowel emptying incomplete', 'Urgency', 'Nocturia', 'Pains', 'Weight',
                   'Sweating', 'Diplopia', 'Remembering', 'Loss of interest', 'Concentrating',
                   'Taste/smelling', 'Hallucinations', 'Delusions', 'Sad, blues', 'Anxiety',
                   'Sex drive', 'Sex difficulty', 'Dizzy', 'Falling', 'Swelling',
                   'Daytime sleepiness', 'Insomnia', 'Intense vivid dreams',
                   'Acting out during dreams', 'Restless legs']

In [6]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 7.7 MB/s eta 0:00:00


PCA

In [7]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE

# Define your classifiers
classifiers = {
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "XGB": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "Extra Trees": ExtraTreesClassifier(),
    "LightGBM": LGBMClassifier(),
    "SVM": SVC(kernel='rbf', C=1, gamma='scale', probability=True),
    "Polynomial Regression": Pipeline([
        ("poly_features", PolynomialFeatures(degree=2)),
        ("linear_regression", LogisticRegression(max_iter=1000))
    ]),
    "MLP": MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42),
    "CatBoost": CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, verbose=0, random_state=42),
}

# Load your data
X = df_filtered[symptom_columns]  # Assuming symptom_columns are your feature columns
y = df_filtered['Rencoded'].apply(lambda x: 2 if x == "Parkinson's" else (1 if x == "Other_Disorders" else 0))

# Standardize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA
pca = PCA(n_components=5)  # Adjust the number of components as needed
X_pca = pca.fit_transform(X_scaled)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, stratify=y, random_state=42)

# Reset indices of X_train and y_train to avoid any index mismatch during cross-validation
X_train_reset = pd.DataFrame(X_train)  # Convert to DataFrame to reset indices
y_train_reset = pd.Series(y_train).reset_index(drop=True)  # Reset index for y_train

# Set up cross-validation
kf = StratifiedKFold(n_splits=10)

# Dictionary to store results
results = {}

# Train each classifier with cross-validation and evaluate
for name, model in classifiers.items():
    print(f"Training {name}...")
    cv_precision = []
    cv_recall = []
    cv_f1 = []

    # Cross-validation loop
    for train_index, val_index in kf.split(X_train_reset, y_train_reset):
        X_fold_train, X_fold_val = X_train_reset.iloc[train_index], X_train_reset.iloc[val_index]
        y_fold_train, y_fold_val = y_train_reset.iloc[train_index], y_train_reset.iloc[val_index]

        # Fit the model
        model.fit(X_fold_train, y_fold_train)

        # Get the classification report for the validation fold
        y_val_pred = model.predict(X_fold_val)
        report = classification_report(y_fold_val, y_val_pred, output_dict=True)

        # Store the weighted precision, recall, and f1-score for each fold
        cv_precision.append(report['weighted avg']['precision'])
        cv_recall.append(report['weighted avg']['recall'])
        cv_f1.append(report['weighted avg']['f1-score'])

    # Calculate average scores over the folds
    results[name] = {
        "Train Precision (Weighted Avg)": np.mean(cv_precision),
        "Train Recall (Weighted Avg)": np.mean(cv_recall),
        "Train F1-Score (Weighted Avg)": np.mean(cv_f1)
    }

    # Test the model on the test set
    model.fit(X_train, y_train)
    y_test_pred = model.predict(X_test)

    # Generate test set classification report and confusion matrix
    test_report = classification_report(y_test, y_test_pred)
    test_confusion_matrix = confusion_matrix(y_test, y_test_pred)

    # Print results
    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {results[name]['Train Precision (Weighted Avg)']:.2f}, "
          f"Recall: {results[name]['Train Recall (Weighted Avg)']:.2f}, "
          f"F1-Score: {results[name]['Train F1-Score (Weighted Avg)']:.2f}")
    print(f"\nTest Set Evaluation for {name}:")
    print(test_report)
    print(f"Confusion Matrix for {name} (Testing):")
    print(test_confusion_matrix)
    print("="*60)


Training KNN...

Weighted Average (Training) for KNN:
Precision: 0.61, Recall: 0.64, F1-Score: 0.61

Test Set Evaluation for KNN:
              precision    recall  f1-score   support

           0       0.57      0.81      0.67        16
           1       0.31      0.17      0.22        23
           2       0.74      0.78      0.76        55

    accuracy                           0.64        94
   macro avg       0.54      0.59      0.55        94
weighted avg       0.61      0.64      0.61        94

Confusion Matrix for KNN (Testing):
[[13  1  2]
 [ 6  4 13]
 [ 4  8 43]]
Training Naive Bayes...

Weighted Average (Training) for Naive Bayes:
Precision: 0.59, Recall: 0.63, F1-Score: 0.58

Test Set Evaluation for Naive Bayes:
              precision    recall  f1-score   support

           0       0.39      0.81      0.53        16
           1       0.67      0.09      0.15        23
           2       0.72      0.76      0.74        55

    accuracy                           0.61 

PCA with Smote

In [8]:
# Define your classifiers
classifiers = {
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "XGB": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "Extra Trees": ExtraTreesClassifier(),
    "LightGBM": LGBMClassifier(),
    'SVM': SVC(random_state=42),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures()), ('model', LogisticRegression(max_iter=1000))])
}

# Apply PCA
pca = PCA(n_components=5)  # Adjust the number of components as needed
X_pca = pca.fit_transform(X_scaled)

# Apply SMOTE to the training data
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Reset indices of X_train_res and y_train_res to avoid any index mismatch during cross-validation
X_train_reset = pd.DataFrame(X_train_res)  # Convert to DataFrame to reset indices
y_train_reset = pd.Series(y_train_res).reset_index(drop=True)  # Reset index for y_train_res

# Set up cross-validation
kf = StratifiedKFold(n_splits=10)

# Dictionary to store results
results = {}

# Train each classifier with cross-validation and evaluate
for name, model in classifiers.items():
    print(f"Training {name}...")
    cv_precision = []
    cv_recall = []
    cv_f1 = []

    # Cross-validation loop
    for train_index, val_index in kf.split(X_train_reset, y_train_reset):
        X_fold_train, X_fold_val = X_train_reset.iloc[train_index], X_train_reset.iloc[val_index]
        y_fold_train, y_fold_val = y_train_reset.iloc[train_index], y_train_reset.iloc[val_index]

        # Fit the model
        model.fit(X_fold_train, y_fold_train)

        # Get the classification report for the validation fold
        y_val_pred = model.predict(X_fold_val)
        report = classification_report(y_fold_val, y_val_pred, output_dict=True)

        # Store the weighted precision, recall, and f1-score for each fold
        cv_precision.append(report['weighted avg']['precision'])
        cv_recall.append(report['weighted avg']['recall'])
        cv_f1.append(report['weighted avg']['f1-score'])

    # Calculate average scores over the folds
    results[name] = {
        "Train Precision (Weighted Avg)": np.mean(cv_precision),
        "Train Recall (Weighted Avg)": np.mean(cv_recall),
        "Train F1-Score (Weighted Avg)": np.mean(cv_f1)
    }

    # Test the model on the test set
    model.fit(X_train_res, y_train_res)
    y_test_pred = model.predict(X_test)

    # Generate test set classification report and confusion matrix
    test_report = classification_report(y_test, y_test_pred)
    test_confusion_matrix = confusion_matrix(y_test, y_test_pred)

    # Print results
    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {results[name]['Train Precision (Weighted Avg)']:.2f}, "
          f"Recall: {results[name]['Train Recall (Weighted Avg)']:.2f}, "
          f"F1-Score: {results[name]['Train F1-Score (Weighted Avg)']:.2f}")
    print(f"\nTest Set Evaluation for {name}:")
    print(test_report)
    print(f"Confusion Matrix for {name} (Testing):")
    print(test_confusion_matrix)
    print("="*60)


Training KNN...

Weighted Average (Training) for KNN:
Precision: 0.76, Recall: 0.74, F1-Score: 0.74

Test Set Evaluation for KNN:
              precision    recall  f1-score   support

           0       0.41      0.75      0.53        16
           1       0.32      0.30      0.31        23
           2       0.79      0.62      0.69        55

    accuracy                           0.56        94
   macro avg       0.51      0.56      0.51        94
weighted avg       0.61      0.56      0.57        94

Confusion Matrix for KNN (Testing):
[[12  3  1]
 [ 8  7  8]
 [ 9 12 34]]
Training Naive Bayes...

Weighted Average (Training) for Naive Bayes:
Precision: 0.59, Recall: 0.60, F1-Score: 0.58

Test Set Evaluation for Naive Bayes:
              precision    recall  f1-score   support

           0       0.36      0.75      0.49        16
           1       0.25      0.22      0.23        23
           2       0.73      0.55      0.62        55

    accuracy                           0.50 

PCA with smote 0.5

In [10]:
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import pandas as pd

# Define classifiers
classifiers = {
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "XGB": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "Extra Trees": ExtraTreesClassifier(),
    "LightGBM": LGBMClassifier(),
    'SVM': SVC(random_state=42),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures()), ('model', LogisticRegression(max_iter=1000))])
}

# Apply PCA
pca = PCA(n_components=5)  # Adjust the number of components as needed
X_pca = pca.fit_transform(X_scaled)

# Calculate sampling strategy for SMOTE
class_counts = y_train.value_counts()
majority_class_count = class_counts.max()

# Define the desired sampling strategy as a dictionary
sampling_strategy = {
    class_label: max(int(majority_class_count * 0.5), count) for class_label, count in class_counts.items()
}

# Apply SMOTE with the sampling strategy
smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Reset indices of X_train_res and y_train_res to avoid any index mismatch during cross-validation
X_train_reset = pd.DataFrame(X_train_res)  # Convert to DataFrame to reset indices
y_train_reset = pd.Series(y_train_res).reset_index(drop=True)  # Reset index for y_train_res

# Set up cross-validation
kf = StratifiedKFold(n_splits=10)

# Dictionary to store results
results = {}

# Train each classifier with cross-validation and evaluate
for name, model in classifiers.items():
    print(f"Training {name}...")
    cv_precision = []
    cv_recall = []
    cv_f1 = []

    # Cross-validation loop
    for train_index, val_index in kf.split(X_train_reset, y_train_reset):
        X_fold_train, X_fold_val = X_train_reset.iloc[train_index], X_train_reset.iloc[val_index]
        y_fold_train, y_fold_val = y_train_reset.iloc[train_index], y_train_reset.iloc[val_index]

        # Fit the model
        model.fit(X_fold_train, y_fold_train)

        # Get the classification report for the validation fold
        y_val_pred = model.predict(X_fold_val)
        report = classification_report(y_fold_val, y_val_pred, output_dict=True)

        # Store the weighted precision, recall, and f1-score for each fold
        cv_precision.append(report['weighted avg']['precision'])
        cv_recall.append(report['weighted avg']['recall'])
        cv_f1.append(report['weighted avg']['f1-score'])

    # Calculate average scores over the folds
    results[name] = {
        "Train Precision (Weighted Avg)": np.mean(cv_precision),
        "Train Recall (Weighted Avg)": np.mean(cv_recall),
        "Train F1-Score (Weighted Avg)": np.mean(cv_f1)
    }

    # Test the model on the test set
    model.fit(X_train_res, y_train_res)
    y_test_pred = model.predict(X_test)

    # Generate test set classification report and confusion matrix
    test_report = classification_report(y_test, y_test_pred)
    test_confusion_matrix = confusion_matrix(y_test, y_test_pred)

    # Print results
    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {results[name]['Train Precision (Weighted Avg)']:.2f}, "
          f"Recall: {results[name]['Train Recall (Weighted Avg)']:.2f}, "
          f"F1-Score: {results[name]['Train F1-Score (Weighted Avg)']:.2f}")
    print(f"\nTest Set Evaluation for {name}:")
    print(test_report)
    print(f"Confusion Matrix for {name} (Testing):")
    print(test_confusion_matrix)
    print("="*60)


Training KNN...

Weighted Average (Training) for KNN:
Precision: 0.64, Recall: 0.66, F1-Score: 0.63

Test Set Evaluation for KNN:
              precision    recall  f1-score   support

           0       0.45      0.81      0.58        16
           1       0.33      0.17      0.23        23
           2       0.75      0.73      0.74        55

    accuracy                           0.61        94
   macro avg       0.51      0.57      0.52        94
weighted avg       0.60      0.61      0.59        94

Confusion Matrix for KNN (Testing):
[[13  2  1]
 [ 7  4 12]
 [ 9  6 40]]
Training Naive Bayes...

Weighted Average (Training) for Naive Bayes:
Precision: 0.55, Recall: 0.63, F1-Score: 0.56

Test Set Evaluation for Naive Bayes:
              precision    recall  f1-score   support

           0       0.37      0.81      0.51        16
           1       0.50      0.09      0.15        23
           2       0.71      0.71      0.71        55

    accuracy                           0.57 

ICA

In [11]:
import numpy as np
import pandas as pd
from sklearn.decomposition import FastICA
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Define your classifiers
classifiers = {
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "XGB": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "Extra Trees": ExtraTreesClassifier(),
    "LightGBM": LGBMClassifier(),
    'SVM': SVC(random_state=42),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures()), ('model', LogisticRegression(max_iter=1000))])
}

# Apply ICA
ica = FastICA(n_components=5)  # Adjust the number of components as needed
X_ica = ica.fit_transform(X_scaled)


# Reset indices of X_train and y_train to avoid any index mismatch during cross-validation
X_train_reset = pd.DataFrame(X_train)  # Convert to DataFrame to reset indices
y_train_reset = pd.Series(y_train).reset_index(drop=True)  # Reset index for y_train

# Set up cross-validation
kf = StratifiedKFold(n_splits=10)

# Dictionary to store results
results = {}

# Train each classifier with cross-validation and evaluate
for name, model in classifiers.items():
    print(f"Training {name}...")
    cv_precision = []
    cv_recall = []
    cv_f1 = []

    # Cross-validation loop
    for train_index, val_index in kf.split(X_train_reset, y_train_reset):
        X_fold_train, X_fold_val = X_train_reset.iloc[train_index], X_train_reset.iloc[val_index]
        y_fold_train, y_fold_val = y_train_reset.iloc[train_index], y_train_reset.iloc[val_index]

        # Fit the model
        model.fit(X_fold_train, y_fold_train)

        # Get the classification report for the validation fold
        y_val_pred = model.predict(X_fold_val)
        report = classification_report(y_fold_val, y_val_pred, output_dict=True)

        # Store the weighted precision, recall, and f1-score for each fold
        cv_precision.append(report['weighted avg']['precision'])
        cv_recall.append(report['weighted avg']['recall'])
        cv_f1.append(report['weighted avg']['f1-score'])

    # Calculate average scores over the folds
    results[name] = {
        "Train Precision (Weighted Avg)": np.mean(cv_precision),
        "Train Recall (Weighted Avg)": np.mean(cv_recall),
        "Train F1-Score (Weighted Avg)": np.mean(cv_f1)
    }

    # Test the model on the test set
    model.fit(X_train, y_train)
    y_test_pred = model.predict(X_test)

    # Generate test set classification report and confusion matrix
    test_report = classification_report(y_test, y_test_pred)
    test_confusion_matrix = confusion_matrix(y_test, y_test_pred)

    # Print results
    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {results[name]['Train Precision (Weighted Avg)']:.2f}, "
          f"Recall: {results[name]['Train Recall (Weighted Avg)']:.2f}, "
          f"F1-Score: {results[name]['Train F1-Score (Weighted Avg)']:.2f}")
    print(f"\nTest Set Evaluation for {name}:")
    print(test_report)
    print(f"Confusion Matrix for {name} (Testing):")
    print(test_confusion_matrix)
    print("="*60)


Training KNN...

Weighted Average (Training) for KNN:
Precision: 0.61, Recall: 0.64, F1-Score: 0.61

Test Set Evaluation for KNN:
              precision    recall  f1-score   support

           0       0.57      0.81      0.67        16
           1       0.31      0.17      0.22        23
           2       0.74      0.78      0.76        55

    accuracy                           0.64        94
   macro avg       0.54      0.59      0.55        94
weighted avg       0.61      0.64      0.61        94

Confusion Matrix for KNN (Testing):
[[13  1  2]
 [ 6  4 13]
 [ 4  8 43]]
Training Naive Bayes...

Weighted Average (Training) for Naive Bayes:
Precision: 0.59, Recall: 0.63, F1-Score: 0.58

Test Set Evaluation for Naive Bayes:
              precision    recall  f1-score   support

           0       0.39      0.81      0.53        16
           1       0.67      0.09      0.15        23
           2       0.72      0.76      0.74        55

    accuracy                           0.61 

ICA with SMOTE

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import FastICA
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Define your classifiers
classifiers = {
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "XGB": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "Extra Trees": ExtraTreesClassifier(),
    "LightGBM": LGBMClassifier(),
    'SVM': SVC(random_state=42),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures()), ('model', LogisticRegression(max_iter=1000))])
}

# Apply ICA
ica = FastICA(n_components=5)  # Adjust the number of components as needed
X_ica = ica.fit_transform(X_scaled)

# Apply SMOTE to the training data
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Reset indices of X_train_res and y_train_res to avoid any index mismatch during cross-validation
X_train_reset = pd.DataFrame(X_train_res)  # Convert to DataFrame to reset indices
y_train_reset = pd.Series(y_train_res).reset_index(drop=True)  # Reset index for y_train_res

# Set up cross-validation
kf = StratifiedKFold(n_splits=10)

# Dictionary to store results
results = {}

# Train each classifier with cross-validation and evaluate
for name, model in classifiers.items():
    print(f"Training {name}...")
    cv_precision = []
    cv_recall = []
    cv_f1 = []

    # Cross-validation loop
    for train_index, val_index in kf.split(X_train_reset, y_train_reset):
        X_fold_train, X_fold_val = X_train_reset.iloc[train_index], X_train_reset.iloc[val_index]
        y_fold_train, y_fold_val = y_train_reset.iloc[train_index], y_train_reset.iloc[val_index]

        # Fit the model
        model.fit(X_fold_train, y_fold_train)

        # Get the classification report for the validation fold
        y_val_pred = model.predict(X_fold_val)
        report = classification_report(y_fold_val, y_val_pred, output_dict=True)

        # Store the weighted precision, recall, and f1-score for each fold
        cv_precision.append(report['weighted avg']['precision'])
        cv_recall.append(report['weighted avg']['recall'])
        cv_f1.append(report['weighted avg']['f1-score'])

    # Calculate average scores over the folds
    results[name] = {
        "Train Precision (Weighted Avg)": np.mean(cv_precision),
        "Train Recall (Weighted Avg)": np.mean(cv_recall),
        "Train F1-Score (Weighted Avg)": np.mean(cv_f1)
    }

    # Test the model on the test set
    model.fit(X_train_res, y_train_res)
    y_test_pred = model.predict(X_test)

    # Generate test set classification report and confusion matrix
    test_report = classification_report(y_test, y_test_pred)
    test_confusion_matrix = confusion_matrix(y_test, y_test_pred)

    # Print results
    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {results[name]['Train Precision (Weighted Avg)']:.2f}, "
          f"Recall: {results[name]['Train Recall (Weighted Avg)']:.2f}, "
          f"F1-Score: {results[name]['Train F1-Score (Weighted Avg)']:.2f}")
    print(f"\nTest Set Evaluation for {name}:")
    print(test_report)
    print(f"Confusion Matrix for {name} (Testing):")
    print(test_confusion_matrix)
    print("="*60)


Training KNN...

Weighted Average (Training) for KNN:
Precision: 0.76, Recall: 0.74, F1-Score: 0.74

Test Set Evaluation for KNN:
              precision    recall  f1-score   support

           0       0.41      0.75      0.53        16
           1       0.32      0.30      0.31        23
           2       0.79      0.62      0.69        55

    accuracy                           0.56        94
   macro avg       0.51      0.56      0.51        94
weighted avg       0.61      0.56      0.57        94

Confusion Matrix for KNN (Testing):
[[12  3  1]
 [ 8  7  8]
 [ 9 12 34]]
Training Naive Bayes...

Weighted Average (Training) for Naive Bayes:
Precision: 0.59, Recall: 0.60, F1-Score: 0.58

Test Set Evaluation for Naive Bayes:
              precision    recall  f1-score   support

           0       0.36      0.75      0.49        16
           1       0.25      0.22      0.23        23
           2       0.73      0.55      0.62        55

    accuracy                           0.50 

ICA with smote 0.5

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import FastICA
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Define your classifiers
classifiers = {
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "XGB": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "Extra Trees": ExtraTreesClassifier(),
    "LightGBM": LGBMClassifier(),
    'SVM': SVC(random_state=42),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures()), ('model', LogisticRegression(max_iter=1000))])
}

# Apply ICA
ica = FastICA(n_components=5)  # Adjust the number of components as needed
X_ica = ica.fit_transform(X_scaled)

# Apply SMOTE to the training data
smote = SMOTE(sampling_strategy=0.5, random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Reset indices of X_train_res and y_train_res to avoid any index mismatch during cross-validation
X_train_reset = pd.DataFrame(X_train_res)  # Convert to DataFrame to reset indices
y_train_reset = pd.Series(y_train_res).reset_index(drop=True)  # Reset index for y_train_res

# Set up cross-validation
kf = StratifiedKFold(n_splits=10)

# Dictionary to store results
results = {}

# Train each classifier with cross-validation and evaluate
for name, model in classifiers.items():
    print(f"Training {name}...")
    cv_precision = []
    cv_recall = []
    cv_f1 = []

    # Cross-validation loop
    for train_index, val_index in kf.split(X_train_reset, y_train_reset):
        X_fold_train, X_fold_val = X_train_reset.iloc[train_index], X_train_reset.iloc[val_index]
        y_fold_train, y_fold_val = y_train_reset.iloc[train_index], y_train_reset.iloc[val_index]

        # Fit the model
        model.fit(X_fold_train, y_fold_train)

        # Get the classification report for the validation fold
        y_val_pred = model.predict(X_fold_val)
        report = classification_report(y_fold_val, y_val_pred, output_dict=True)

        # Store the weighted precision, recall, and f1-score for each fold
        cv_precision.append(report['weighted avg']['precision'])
        cv_recall.append(report['weighted avg']['recall'])
        cv_f1.append(report['weighted avg']['f1-score'])

    # Calculate average scores over the folds
    results[name] = {
        "Train Precision (Weighted Avg)": np.mean(cv_precision),
        "Train Recall (Weighted Avg)": np.mean(cv_recall),
        "Train F1-Score (Weighted Avg)": np.mean(cv_f1)
    }

    # Test the model on the test set
    model.fit(X_train_res, y_train_res)
    y_test_pred = model.predict(X_test)

    # Generate test set classification report and confusion matrix
    test_report = classification_report(y_test, y_test_pred)
    test_confusion_matrix = confusion_matrix(y_test, y_test_pred)

    # Print results
    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {results[name]['Train Precision (Weighted Avg)']:.2f}, "
          f"Recall: {results[name]['Train Recall (Weighted Avg)']:.2f}, "
          f"F1-Score: {results[name]['Train F1-Score (Weighted Avg)']:.2f}")
    print(f"\nTest Set Evaluation for {name}:")
    print(test_report)
    print(f"Confusion Matrix for {name} (Testing):")
    print(test_confusion_matrix)
    print("="*60)

ANOVA (f_classif),Forward Selection, Backward Selection, Recursive Feature Elimination (RFE)

In [14]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif, chi2, RFE, SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Define classifiers
classifiers = {
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "XGB": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "Extra Trees": ExtraTreesClassifier(),
    'SVM': SVC(random_state=42),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures()), ('model', LogisticRegression(max_iter=1000))])
}


# Set up cross-validation
kf = StratifiedKFold(n_splits=10)
results = {}

# Define feature selection methods
feature_selection_methods = {
    "ANOVA (f_classif)": SelectKBest(f_classif, k=6),  # Adjust k as needed
    "Forward Selection": SequentialFeatureSelector(LogisticRegression(), n_features_to_select=6, direction='forward', cv=kf),
    "Backward Selection": SequentialFeatureSelector(LogisticRegression(), n_features_to_select=6, direction='backward', cv=kf),
    "Recursive Feature Elimination (RFE)": RFE(estimator=RandomForestClassifier(), n_features_to_select=7)  # Adjust n_features_to_select
}

# Apply each feature selection method
for method_name, selector in feature_selection_methods.items():
    print(f"\nUsing Feature Selection Method: {method_name}")

    # Perform feature selection
    if method_name in ["Forward Selection", "Backward Selection"]:
        # These methods require a fitted selector
        selector.fit(X_scaled, y)
        X_selected = selector.transform(X_scaled)
    else:
        X_selected = selector.fit_transform(X_scaled, y)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, stratify=y, random_state=42)
    X_train_reset = pd.DataFrame(X_train).reset_index(drop=True)
    y_train_reset = pd.Series(y_train).reset_index(drop=True)

    # Train and evaluate classifiers
    for name, model in classifiers.items():
        print(f"\nTraining {name} with {method_name}...")

        cv_precision = []
        cv_recall = []
        cv_f1 = []

        for train_index, val_index in kf.split(X_train_reset, y_train_reset):
            X_fold_train, X_fold_val = X_train_reset.iloc[train_index], X_train_reset.iloc[val_index]
            y_fold_train, y_fold_val = y_train_reset.iloc[train_index], y_train_reset.iloc[val_index]

            # Fit model on training fold
            model.fit(X_fold_train, y_fold_train)

            # Validation fold evaluation
            y_val_pred = model.predict(X_fold_val)
            report = classification_report(y_fold_val, y_val_pred, output_dict=True)

            # Store metrics
            cv_precision.append(report['weighted avg']['precision'])
            cv_recall.append(report['weighted avg']['recall'])
            cv_f1.append(report['weighted avg']['f1-score'])

        # Average scores
        results[(method_name, name)] = {
            "Train Precision (Weighted Avg)": np.mean(cv_precision),
            "Train Recall (Weighted Avg)": np.mean(cv_recall),
            "Train F1-Score (Weighted Avg)": np.mean(cv_f1)
        }

        # Test set evaluation
        model.fit(X_train, y_train)
        y_test_pred = model.predict(X_test)
        test_report = classification_report(y_test, y_test_pred)
        test_confusion_matrix = confusion_matrix(y_test, y_test_pred)

        # Print results
        print(f"\nWeighted Average (Training) for {name} using {method_name}:")
        print(f"Precision: {results[(method_name, name)]['Train Precision (Weighted Avg)']:.2f}, "
              f"Recall: {results[(method_name, name)]['Train Recall (Weighted Avg)']:.2f}, "
              f"F1-Score: {results[(method_name, name)]['Train F1-Score (Weighted Avg)']:.2f}")
        print(f"\nTest Set Evaluation for {name} using {method_name}:")
        print(test_report)
        print(f"Confusion Matrix for {name} (Testing) using {method_name}:")
        print(test_confusion_matrix)
        print("="*60)



Using Feature Selection Method: ANOVA (f_classif)

Training KNN with ANOVA (f_classif)...

Weighted Average (Training) for KNN using ANOVA (f_classif):
Precision: 0.58, Recall: 0.63, F1-Score: 0.60

Test Set Evaluation for KNN using ANOVA (f_classif):
              precision    recall  f1-score   support

           0       0.20      0.06      0.10        16
           1       0.32      0.43      0.37        23
           2       0.72      0.76      0.74        55

    accuracy                           0.56        94
   macro avg       0.42      0.42      0.40        94
weighted avg       0.54      0.56      0.54        94

Confusion Matrix for KNN (Testing) using ANOVA (f_classif):
[[ 1 11  4]
 [ 1 10 12]
 [ 3 10 42]]

Training Naive Bayes with ANOVA (f_classif)...

Weighted Average (Training) for Naive Bayes using ANOVA (f_classif):
Precision: 0.62, Recall: 0.56, F1-Score: 0.53

Test Set Evaluation for Naive Bayes using ANOVA (f_classif):
              precision    recall  f1-score

ANOVA (f_classif)

In [8]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif, chi2, RFE, SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Define classifiers
classifiers = {
    "LightGBM": LGBMClassifier()
}


# Set up cross-validation
kf = StratifiedKFold(n_splits=10)
results = {}

# Define feature selection methods
feature_selection_methods = {
    "ANOVA (f_classif)": SelectKBest(f_classif, k=6),  # Adjust k as needed
}

# Apply each feature selection method
for method_name, selector in feature_selection_methods.items():
    print(f"\nUsing Feature Selection Method: {method_name}")

    # Perform feature selection
    if method_name in ["Forward Selection", "Backward Selection"]:
        # These methods require a fitted selector
        selector.fit(X_scaled, y)
        X_selected = selector.transform(X_scaled)
    else:
        X_selected = selector.fit_transform(X_scaled, y)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, stratify=y, random_state=42)
    X_train_reset = pd.DataFrame(X_train).reset_index(drop=True)
    y_train_reset = pd.Series(y_train).reset_index(drop=True)

    # Train and evaluate classifiers
    for name, model in classifiers.items():
        print(f"\nTraining {name} with {method_name}...")

        cv_precision = []
        cv_recall = []
        cv_f1 = []

        for train_index, val_index in kf.split(X_train_reset, y_train_reset):
            X_fold_train, X_fold_val = X_train_reset.iloc[train_index], X_train_reset.iloc[val_index]
            y_fold_train, y_fold_val = y_train_reset.iloc[train_index], y_train_reset.iloc[val_index]

            # Fit model on training fold
            model.fit(X_fold_train, y_fold_train)

            # Validation fold evaluation
            y_val_pred = model.predict(X_fold_val)
            report = classification_report(y_fold_val, y_val_pred, output_dict=True)

            # Store metrics
            cv_precision.append(report['weighted avg']['precision'])
            cv_recall.append(report['weighted avg']['recall'])
            cv_f1.append(report['weighted avg']['f1-score'])

        # Average scores
        results[(method_name, name)] = {
            "Train Precision (Weighted Avg)": np.mean(cv_precision),
            "Train Recall (Weighted Avg)": np.mean(cv_recall),
            "Train F1-Score (Weighted Avg)": np.mean(cv_f1)
        }

        # Test set evaluation
        model.fit(X_train, y_train)
        y_test_pred = model.predict(X_test)
        test_report = classification_report(y_test, y_test_pred)
        test_confusion_matrix = confusion_matrix(y_test, y_test_pred)

        # Print results
        print(f"\nWeighted Average (Training) for {name} using {method_name}:")
        print(f"Precision: {results[(method_name, name)]['Train Precision (Weighted Avg)']:.2f}, "
              f"Recall: {results[(method_name, name)]['Train Recall (Weighted Avg)']:.2f}, "
              f"F1-Score: {results[(method_name, name)]['Train F1-Score (Weighted Avg)']:.2f}")
        print(f"\nTest Set Evaluation for {name} using {method_name}:")
        print(test_report)
        print(f"Confusion Matrix for {name} (Testing) using {method_name}:")
        print(test_confusion_matrix)
        print("="*60)



Using Feature Selection Method: ANOVA (f_classif)

Training LightGBM with ANOVA (f_classif)...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000163 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 18
[LightGBM] [Info] Number of data points in the train set: 337, number of used features: 6
[LightGBM] [Info] Start training from score -1.777032
[LightGBM] [Info] Start training from score -1.425634
[LightGBM] [Info] Start training from score -0.526778
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning

Forward Selection

In [9]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif, chi2, RFE, SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Define classifiers
classifiers = {
    "LightGBM": LGBMClassifier()
}


# Set up cross-validation
kf = StratifiedKFold(n_splits=10)
results = {}

# Define feature selection methods
feature_selection_methods = {
    "Forward Selection": SequentialFeatureSelector(LogisticRegression(), n_features_to_select=6, direction='forward', cv=kf),
}

# Apply each feature selection method
for method_name, selector in feature_selection_methods.items():
    print(f"\nUsing Feature Selection Method: {method_name}")

    # Perform feature selection
    if method_name in ["Forward Selection", "Backward Selection"]:
        # These methods require a fitted selector
        selector.fit(X_scaled, y)
        X_selected = selector.transform(X_scaled)
    else:
        X_selected = selector.fit_transform(X_scaled, y)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, stratify=y, random_state=42)
    X_train_reset = pd.DataFrame(X_train).reset_index(drop=True)
    y_train_reset = pd.Series(y_train).reset_index(drop=True)

    # Train and evaluate classifiers
    for name, model in classifiers.items():
        print(f"\nTraining {name} with {method_name}...")

        cv_precision = []
        cv_recall = []
        cv_f1 = []

        for train_index, val_index in kf.split(X_train_reset, y_train_reset):
            X_fold_train, X_fold_val = X_train_reset.iloc[train_index], X_train_reset.iloc[val_index]
            y_fold_train, y_fold_val = y_train_reset.iloc[train_index], y_train_reset.iloc[val_index]

            # Fit model on training fold
            model.fit(X_fold_train, y_fold_train)

            # Validation fold evaluation
            y_val_pred = model.predict(X_fold_val)
            report = classification_report(y_fold_val, y_val_pred, output_dict=True)

            # Store metrics
            cv_precision.append(report['weighted avg']['precision'])
            cv_recall.append(report['weighted avg']['recall'])
            cv_f1.append(report['weighted avg']['f1-score'])

        # Average scores
        results[(method_name, name)] = {
            "Train Precision (Weighted Avg)": np.mean(cv_precision),
            "Train Recall (Weighted Avg)": np.mean(cv_recall),
            "Train F1-Score (Weighted Avg)": np.mean(cv_f1)
        }

        # Test set evaluation
        model.fit(X_train, y_train)
        y_test_pred = model.predict(X_test)
        test_report = classification_report(y_test, y_test_pred)
        test_confusion_matrix = confusion_matrix(y_test, y_test_pred)

        # Print results
        print(f"\nWeighted Average (Training) for {name} using {method_name}:")
        print(f"Precision: {results[(method_name, name)]['Train Precision (Weighted Avg)']:.2f}, "
              f"Recall: {results[(method_name, name)]['Train Recall (Weighted Avg)']:.2f}, "
              f"F1-Score: {results[(method_name, name)]['Train F1-Score (Weighted Avg)']:.2f}")
        print(f"\nTest Set Evaluation for {name} using {method_name}:")
        print(test_report)
        print(f"Confusion Matrix for {name} (Testing) using {method_name}:")
        print(test_confusion_matrix)
        print("="*60)



Using Feature Selection Method: Forward Selection

Training LightGBM with Forward Selection...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000021 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 18
[LightGBM] [Info] Number of data points in the train set: 337, number of used features: 6
[LightGBM] [Info] Start training from score -1.777032
[LightGBM] [Info] Start training from score -1.425634
[LightGBM] [Info] Start training from score -0.526778
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning

Backward Selection

In [10]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif, chi2, RFE, SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Define classifiers
classifiers = {
    "LightGBM": LGBMClassifier()
}


# Set up cross-validation
kf = StratifiedKFold(n_splits=10)
results = {}

# Define feature selection methods
feature_selection_methods = {
    "Backward Selection": SequentialFeatureSelector(LogisticRegression(), n_features_to_select=6, direction='backward', cv=kf),
}

# Apply each feature selection method
for method_name, selector in feature_selection_methods.items():
    print(f"\nUsing Feature Selection Method: {method_name}")

    # Perform feature selection
    if method_name in ["Forward Selection", "Backward Selection"]:
        # These methods require a fitted selector
        selector.fit(X_scaled, y)
        X_selected = selector.transform(X_scaled)
    else:
        X_selected = selector.fit_transform(X_scaled, y)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, stratify=y, random_state=42)
    X_train_reset = pd.DataFrame(X_train).reset_index(drop=True)
    y_train_reset = pd.Series(y_train).reset_index(drop=True)

    # Train and evaluate classifiers
    for name, model in classifiers.items():
        print(f"\nTraining {name} with {method_name}...")

        cv_precision = []
        cv_recall = []
        cv_f1 = []

        for train_index, val_index in kf.split(X_train_reset, y_train_reset):
            X_fold_train, X_fold_val = X_train_reset.iloc[train_index], X_train_reset.iloc[val_index]
            y_fold_train, y_fold_val = y_train_reset.iloc[train_index], y_train_reset.iloc[val_index]

            # Fit model on training fold
            model.fit(X_fold_train, y_fold_train)

            # Validation fold evaluation
            y_val_pred = model.predict(X_fold_val)
            report = classification_report(y_fold_val, y_val_pred, output_dict=True)

            # Store metrics
            cv_precision.append(report['weighted avg']['precision'])
            cv_recall.append(report['weighted avg']['recall'])
            cv_f1.append(report['weighted avg']['f1-score'])

        # Average scores
        results[(method_name, name)] = {
            "Train Precision (Weighted Avg)": np.mean(cv_precision),
            "Train Recall (Weighted Avg)": np.mean(cv_recall),
            "Train F1-Score (Weighted Avg)": np.mean(cv_f1)
        }

        # Test set evaluation
        model.fit(X_train, y_train)
        y_test_pred = model.predict(X_test)
        test_report = classification_report(y_test, y_test_pred)
        test_confusion_matrix = confusion_matrix(y_test, y_test_pred)

        # Print results
        print(f"\nWeighted Average (Training) for {name} using {method_name}:")
        print(f"Precision: {results[(method_name, name)]['Train Precision (Weighted Avg)']:.2f}, "
              f"Recall: {results[(method_name, name)]['Train Recall (Weighted Avg)']:.2f}, "
              f"F1-Score: {results[(method_name, name)]['Train F1-Score (Weighted Avg)']:.2f}")
        print(f"\nTest Set Evaluation for {name} using {method_name}:")
        print(test_report)
        print(f"Confusion Matrix for {name} (Testing) using {method_name}:")
        print(test_confusion_matrix)
        print("="*60)


Using Feature Selection Method: Backward Selection

Training LightGBM with Backward Selection...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000066 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 18
[LightGBM] [Info] Number of data points in the train set: 337, number of used features: 6
[LightGBM] [Info] Start training from score -1.777032
[LightGBM] [Info] Start training from score -1.425634
[LightGBM] [Info] Start training from score -0.526778
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warni

Recursive Feature Elimination (RFE)

In [11]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif, chi2, RFE, SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Define classifiers
classifiers = {
    "LightGBM": LGBMClassifier()
}


# Set up cross-validation
kf = StratifiedKFold(n_splits=10)
results = {}

# Define feature selection methods
feature_selection_methods = {
    "Recursive Feature Elimination (RFE)": RFE(estimator=RandomForestClassifier(), n_features_to_select=7)  # Adjust n_features_to_select
}

# Apply each feature selection method
for method_name, selector in feature_selection_methods.items():
    print(f"\nUsing Feature Selection Method: {method_name}")

    # Perform feature selection
    if method_name in ["Forward Selection", "Backward Selection"]:
        # These methods require a fitted selector
        selector.fit(X_scaled, y)
        X_selected = selector.transform(X_scaled)
    else:
        X_selected = selector.fit_transform(X_scaled, y)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, stratify=y, random_state=42)
    X_train_reset = pd.DataFrame(X_train).reset_index(drop=True)
    y_train_reset = pd.Series(y_train).reset_index(drop=True)

    # Train and evaluate classifiers
    for name, model in classifiers.items():
        print(f"\nTraining {name} with {method_name}...")

        cv_precision = []
        cv_recall = []
        cv_f1 = []

        for train_index, val_index in kf.split(X_train_reset, y_train_reset):
            X_fold_train, X_fold_val = X_train_reset.iloc[train_index], X_train_reset.iloc[val_index]
            y_fold_train, y_fold_val = y_train_reset.iloc[train_index], y_train_reset.iloc[val_index]

            # Fit model on training fold
            model.fit(X_fold_train, y_fold_train)

            # Validation fold evaluation
            y_val_pred = model.predict(X_fold_val)
            report = classification_report(y_fold_val, y_val_pred, output_dict=True)

            # Store metrics
            cv_precision.append(report['weighted avg']['precision'])
            cv_recall.append(report['weighted avg']['recall'])
            cv_f1.append(report['weighted avg']['f1-score'])

        # Average scores
        results[(method_name, name)] = {
            "Train Precision (Weighted Avg)": np.mean(cv_precision),
            "Train Recall (Weighted Avg)": np.mean(cv_recall),
            "Train F1-Score (Weighted Avg)": np.mean(cv_f1)
        }

        # Test set evaluation
        model.fit(X_train, y_train)
        y_test_pred = model.predict(X_test)
        test_report = classification_report(y_test, y_test_pred)
        test_confusion_matrix = confusion_matrix(y_test, y_test_pred)

        # Print results
        print(f"\nWeighted Average (Training) for {name} using {method_name}:")
        print(f"Precision: {results[(method_name, name)]['Train Precision (Weighted Avg)']:.2f}, "
              f"Recall: {results[(method_name, name)]['Train Recall (Weighted Avg)']:.2f}, "
              f"F1-Score: {results[(method_name, name)]['Train F1-Score (Weighted Avg)']:.2f}")
        print(f"\nTest Set Evaluation for {name} using {method_name}:")
        print(test_report)
        print(f"Confusion Matrix for {name} (Testing) using {method_name}:")
        print(test_confusion_matrix)
        print("="*60)



Using Feature Selection Method: Recursive Feature Elimination (RFE)

Training LightGBM with Recursive Feature Elimination (RFE)...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000049 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21
[LightGBM] [Info] Number of data points in the train set: 337, number of used features: 7
[LightGBM] [Info] Start training from score -1.777032
[LightGBM] [Info] Start training from score -1.425634
[LightGBM] [Info] Start training from score -0.526778
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with pos

ANOVA (f_classif) with smote

In [16]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier

# Define classifiers
classifiers = {
    "LightGBM": LGBMClassifier()
}

# Set up cross-validation
kf = StratifiedKFold(n_splits=10)
results = {}

# Define feature selection methods
feature_selection_methods = {
    "ANOVA (f_classif)": SelectKBest(f_classif, k=6),  # Adjust k as needed
}

# Apply each feature selection method
for method_name, selector in feature_selection_methods.items():
    print(f"\nUsing Feature Selection Method: {method_name}")

    # Perform feature selection
    X_selected = selector.fit_transform(X, y)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, stratify=y, random_state=42)
    X_train_reset = pd.DataFrame(X_train).reset_index(drop=True)
    y_train_reset = pd.Series(y_train).reset_index(drop=True)

    # Apply SMOTE for the training set
    smote = SMOTE(random_state=42)
    X_train_smote, y_train_smote = smote.fit_resample(X_train_reset, y_train_reset)

    # Train and evaluate classifiers
    for name, model in classifiers.items():
        print(f"\nTraining {name} with {method_name}...")

        cv_precision = []
        cv_recall = []
        cv_f1 = []

        for train_index, val_index in kf.split(X_train_smote, y_train_smote):
            X_fold_train, X_fold_val = X_train_smote.iloc[train_index], X_train_smote.iloc[val_index]
            y_fold_train, y_fold_val = y_train_smote.iloc[train_index], y_train_smote.iloc[val_index]

            # Fit model on training fold
            model.fit(X_fold_train, y_fold_train)

            # Validation fold evaluation
            y_val_pred = model.predict(X_fold_val)
            report = classification_report(y_fold_val, y_val_pred, output_dict=True)

            # Store metrics
            cv_precision.append(report['weighted avg']['precision'])
            cv_recall.append(report['weighted avg']['recall'])
            cv_f1.append(report['weighted avg']['f1-score'])

        # Average scores
        results[(method_name, name)] = {
            "Train Precision (Weighted Avg)": np.mean(cv_precision),
            "Train Recall (Weighted Avg)": np.mean(cv_recall),
            "Train F1-Score (Weighted Avg)": np.mean(cv_f1)
        }

        # Test set evaluation
        model.fit(X_train_smote, y_train_smote)
        y_test_pred = model.predict(X_test)
        test_report = classification_report(y_test, y_test_pred)
        test_confusion_matrix = confusion_matrix(y_test, y_test_pred)

        # Print results
        print(f"\nWeighted Average (Training) for {name} using {method_name}:")
        print(f"Precision: {results[(method_name, name)]['Train Precision (Weighted Avg)']:.2f}, "
              f"Recall: {results[(method_name, name)]['Train Recall (Weighted Avg)']:.2f}, "
              f"F1-Score: {results[(method_name, name)]['Train F1-Score (Weighted Avg)']:.2f}")
        print(f"\nTest Set Evaluation for {name} using {method_name}:")
        print(test_report)
        print(f"Confusion Matrix for {name} (Testing) using {method_name}:")
        print(test_confusion_matrix)
        print("="*60)



Using Feature Selection Method: ANOVA (f_classif)

Training LightGBM with ANOVA (f_classif)...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000095 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12
[LightGBM] [Info] Number of data points in the train set: 596, number of used features: 6
[LightGBM] [Info] Start training from score -1.096936
[LightGBM] [Info] Start training from score -1.101974
[LightGBM] [Info] Start training from score -1.096936
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning

Forward selction with smote(light)

In [15]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier

# Define classifiers
classifiers = {
    "LightGBM": LGBMClassifier()
}

# Set up cross-validation
kf = StratifiedKFold(n_splits=10)
results = {}

# Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Standardize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define the base model for feature selection (can be any classifier)
base_model = RandomForestClassifier(random_state=42)

# Apply Forward Selection using SequentialFeatureSelector
print("\nUsing Forward Selection for Feature Selection")
sfs = SequentialFeatureSelector(base_model, n_features_to_select=6, direction='forward', scoring='accuracy', cv=5, n_jobs=-1)
X_train_selected = sfs.fit_transform(X_train_scaled, y_train)
X_test_selected = sfs.transform(X_test_scaled)

# Apply SMOTE to the selected features in the training set
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_selected, y_train)

# Train and evaluate classifiers
for name, model in classifiers.items():
    print(f"\nTraining {name} with Forward Selection and SMOTE...")

    cv_precision = []
    cv_recall = []
    cv_f1 = []

    for train_index, val_index in kf.split(X_train_smote, y_train_smote):
        X_fold_train, X_fold_val = X_train_smote[train_index], X_train_smote[val_index]
        y_fold_train, y_fold_val = y_train_smote[train_index], y_train_smote[val_index]

        # Fit model on training fold
        model.fit(X_fold_train, y_fold_train)

        # Validation fold evaluation
        y_val_pred = model.predict(X_fold_val)
        report = classification_report(y_fold_val, y_val_pred, output_dict=True)

        # Store metrics
        cv_precision.append(report['weighted avg']['precision'])
        cv_recall.append(report['weighted avg']['recall'])
        cv_f1.append(report['weighted avg']['f1-score'])

    # Average scores
    results[(name)] = {
        "Train Precision (Weighted Avg)": np.mean(cv_precision),
        "Train Recall (Weighted Avg)": np.mean(cv_recall),
        "Train F1-Score (Weighted Avg)": np.mean(cv_f1)
    }

    # Test set evaluation
    model.fit(X_train_smote, y_train_smote)
    y_test_pred = model.predict(X_test_selected)
    test_report = classification_report(y_test, y_test_pred)
    test_confusion_matrix = confusion_matrix(y_test, y_test_pred)

    # Print results
    print(f"\nWeighted Average (Training) for {name} with Forward Selection and SMOTE:")
    print(f"Precision: {results[(name)]['Train Precision (Weighted Avg)']:.2f}, "
          f"Recall: {results[(name)]['Train Recall (Weighted Avg)']:.2f}, "
          f"F1-Score: {results[(name)]['Train F1-Score (Weighted Avg)']:.2f}")
    print(f"\nTest Set Evaluation for {name} with Forward Selection and SMOTE:")
    print(test_report)
    print(f"Confusion Matrix for {name} (Testing) with Forward Selection and SMOTE:")
    print(test_confusion_matrix)
    print("="*60)



Using Forward Selection for Feature Selection

Training LightGBM with Forward Selection and SMOTE...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000105 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 47
[LightGBM] [Info] Number of data points in the train set: 596, number of used features: 6
[LightGBM] [Info] Start training from score -1.096936
[LightGBM] [Info] Start training from score -1.101974
[LightGBM] [Info] Start training from score -1.096936
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [W

Backward with smote(light)

In [14]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier

# Define classifiers
classifiers = {
    "LightGBM": LGBMClassifier()
}

# Set up cross-validation
kf = StratifiedKFold(n_splits=10)
results = {}

# Define the backward selection method
print("\nUsing Feature Selection Method: Backward Selection")

# Set up a base model for feature selection
base_model = RandomForestClassifier(random_state=42)

# Apply backward selection using SequentialFeatureSelector
selector = SequentialFeatureSelector(base_model, n_features_to_select=5, direction='backward', cv=5)
X_selected = selector.fit_transform(X, y)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, stratify=y, random_state=42)
X_train_reset = pd.DataFrame(X_train).reset_index(drop=True)
y_train_reset = pd.Series(y_train).reset_index(drop=True)

# Apply SMOTE for the training set
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_reset, y_train_reset)

# Train and evaluate classifiers
for name, model in classifiers.items():
    print(f"\nTraining {name} with Backward Selection...")

    cv_precision = []
    cv_recall = []
    cv_f1 = []

    for train_index, val_index in kf.split(X_train_smote, y_train_smote):
        X_fold_train, X_fold_val = X_train_smote.iloc[train_index], X_train_smote.iloc[val_index]
        y_fold_train, y_fold_val = y_train_smote.iloc[train_index], y_train_smote.iloc[val_index]

        # Fit model on training fold
        model.fit(X_fold_train, y_fold_train)

        # Validation fold evaluation
        y_val_pred = model.predict(X_fold_val)
        report = classification_report(y_fold_val, y_val_pred, output_dict=True)

        # Store metrics
        cv_precision.append(report['weighted avg']['precision'])
        cv_recall.append(report['weighted avg']['recall'])
        cv_f1.append(report['weighted avg']['f1-score'])

    # Average scores
    results[(name, "Backward Selection")] = {
        "Train Precision (Weighted Avg)": np.mean(cv_precision),
        "Train Recall (Weighted Avg)": np.mean(cv_recall),
        "Train F1-Score (Weighted Avg)": np.mean(cv_f1)
    }

    # Test set evaluation
    model.fit(X_train_smote, y_train_smote)
    y_test_pred = model.predict(X_test)
    test_report = classification_report(y_test, y_test_pred)
    test_confusion_matrix = confusion_matrix(y_test, y_test_pred)

    # Print results
    print(f"\nWeighted Average (Training) for {name} using Backward Selection:")
    print(f"Precision: {results[(name, 'Backward Selection')]['Train Precision (Weighted Avg)']:.2f}, "
          f"Recall: {results[(name, 'Backward Selection')]['Train Recall (Weighted Avg)']:.2f}, "
          f"F1-Score: {results[(name, 'Backward Selection')]['Train F1-Score (Weighted Avg)']:.2f}")
    print(f"\nTest Set Evaluation for {name} using Backward Selection:")
    print(test_report)
    print(f"Confusion Matrix for {name} (Testing) using Backward Selection:")
    print(test_confusion_matrix)
    print("="*60)



Using Feature Selection Method: Backward Selection

Training LightGBM with Backward Selection...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000158 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 10
[LightGBM] [Info] Number of data points in the train set: 596, number of used features: 5
[LightGBM] [Info] Start training from score -1.096936
[LightGBM] [Info] Start training from score -1.101974
[LightGBM] [Info] Start training from score -1.096936
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warni

RFE with smote(light)

In [15]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier

# Define classifiers
classifiers = {
    "LightGBM": LGBMClassifier()
}

# Set up cross-validation
kf = StratifiedKFold(n_splits=10)
results = {}

# Define the RFE feature selection method
print("\nUsing Feature Selection Method: RFE")

# Set up a base model for RFE
base_model = RandomForestClassifier(random_state=42)

# Apply RFE with the base model to select top features
rfe_selector = RFE(estimator=base_model, n_features_to_select=7)  # Adjust n_features_to_select as needed
X_selected = rfe_selector.fit_transform(X, y)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, stratify=y, random_state=42)
X_train_reset = pd.DataFrame(X_train).reset_index(drop=True)
y_train_reset = pd.Series(y_train).reset_index(drop=True)

# Apply SMOTE for the training set
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_reset, y_train_reset)

# Train and evaluate classifiers
for name, model in classifiers.items():
    print(f"\nTraining {name} with RFE...")

    cv_precision = []
    cv_recall = []
    cv_f1 = []

    for train_index, val_index in kf.split(X_train_smote, y_train_smote):
        X_fold_train, X_fold_val = X_train_smote.iloc[train_index], X_train_smote.iloc[val_index]
        y_fold_train, y_fold_val = y_train_smote.iloc[train_index], y_train_smote.iloc[val_index]

        # Fit model on training fold
        model.fit(X_fold_train, y_fold_train)

        # Validation fold evaluation
        y_val_pred = model.predict(X_fold_val)
        report = classification_report(y_fold_val, y_val_pred, output_dict=True)

        # Store metrics
        cv_precision.append(report['weighted avg']['precision'])
        cv_recall.append(report['weighted avg']['recall'])
        cv_f1.append(report['weighted avg']['f1-score'])

    # Average scores
    results[(name, "RFE")] = {
        "Train Precision (Weighted Avg)": np.mean(cv_precision),
        "Train Recall (Weighted Avg)": np.mean(cv_recall),
        "Train F1-Score (Weighted Avg)": np.mean(cv_f1)
    }

    # Test set evaluation
    model.fit(X_train_smote, y_train_smote)
    y_test_pred = model.predict(X_test)
    test_report = classification_report(y_test, y_test_pred)
    test_confusion_matrix = confusion_matrix(y_test, y_test_pred)

    # Print results
    print(f"\nWeighted Average (Training) for {name} using RFE:")
    print(f"Precision: {results[(name, 'RFE')]['Train Precision (Weighted Avg)']:.2f}, "
          f"Recall: {results[(name, 'RFE')]['Train Recall (Weighted Avg)']:.2f}, "
          f"F1-Score: {results[(name, 'RFE')]['Train F1-Score (Weighted Avg)']:.2f}")
    print(f"\nTest Set Evaluation for {name} using RFE:")
    print(test_report)
    print(f"Confusion Matrix for {name} (Testing) using RFE:")
    print(test_confusion_matrix)
    print("="*60)



Using Feature Selection Method: RFE

Training LightGBM with RFE...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000275 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 14
[LightGBM] [Info] Number of data points in the train set: 596, number of used features: 7
[LightGBM] [Info] Start training from score -1.096936
[LightGBM] [Info] Start training from score -1.101974
[LightGBM] [Info] Start training from score -1.096936
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with pos

Annova, forward, backward and RFE with smote(9 classifiers)

In [15]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif, chi2, RFE, SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE

# Define classifiers
classifiers = {
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "XGB": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "Extra Trees": ExtraTreesClassifier(),
    'SVM': SVC(random_state=42),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures()), ('model', LogisticRegression(max_iter=1000))])
}


# Set up cross-validation
kf = StratifiedKFold(n_splits=10)
results = {}

# Define feature selection methods
feature_selection_methods = {
    "ANOVA (f_classif)": SelectKBest(f_classif, k=6),  # Adjust k as needed
    "Forward Selection": SequentialFeatureSelector(LogisticRegression(), n_features_to_select=6, direction='forward', cv=kf),
    "Backward Selection": SequentialFeatureSelector(LogisticRegression(), n_features_to_select=5, direction='backward', cv=kf),
    "Recursive Feature Elimination (RFE)": RFE(estimator=RandomForestClassifier(), n_features_to_select=7)  # Adjust n_features_to_select
}

# Apply each feature selection method
for method_name, selector in feature_selection_methods.items():
    print(f"\nUsing Feature Selection Method: {method_name}")

    # Perform feature selection
    if method_name in ["Forward Selection", "Backward Selection"]:
        selector.fit(X_scaled, y)
        X_selected = selector.transform(X_scaled)
    else:
        X_selected = selector.fit_transform(X_scaled, y)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, stratify=y, random_state=42)
    X_train_reset = pd.DataFrame(X_train).reset_index(drop=True)
    y_train_reset = pd.Series(y_train).reset_index(drop=True)

    # Initialize SMOTE
    smote = SMOTE(random_state=42)
    X_train_sm, y_train_sm = smote.fit_resample(X_train_reset, y_train_reset)

    # Train and evaluate classifiers
    for name, model in classifiers.items():
        print(f"\nTraining {name} with {method_name} and SMOTE...")

        cv_precision = []
        cv_recall = []
        cv_f1 = []

        for train_index, val_index in kf.split(X_train_sm, y_train_sm):
            X_fold_train, X_fold_val = X_train_sm.iloc[train_index], X_train_sm.iloc[val_index]
            y_fold_train, y_fold_val = y_train_sm.iloc[train_index], y_train_sm.iloc[val_index]

            # Fit model on SMOTE-resampled training fold
            model.fit(X_fold_train, y_fold_train)

            # Validation fold evaluation
            y_val_pred = model.predict(X_fold_val)
            report = classification_report(y_fold_val, y_val_pred, output_dict=True)

            # Store metrics
            cv_precision.append(report['weighted avg']['precision'])
            cv_recall.append(report['weighted avg']['recall'])
            cv_f1.append(report['weighted avg']['f1-score'])

        # Average scores
        results[(method_name, name)] = {
            "Train Precision (Weighted Avg)": np.mean(cv_precision),
            "Train Recall (Weighted Avg)": np.mean(cv_recall),
            "Train F1-Score (Weighted Avg)": np.mean(cv_f1)
        }

        # Test set evaluation
        model.fit(X_train_sm, y_train_sm)
        y_test_pred = model.predict(X_test)
        test_report = classification_report(y_test, y_test_pred)
        test_confusion_matrix = confusion_matrix(y_test, y_test_pred)

        # Print results
        print(f"\nWeighted Average (Training) for {name} using {method_name} with SMOTE:")
        print(f"Precision: {results[(method_name, name)]['Train Precision (Weighted Avg)']:.2f}, "
              f"Recall: {results[(method_name, name)]['Train Recall (Weighted Avg)']:.2f}, "
              f"F1-Score: {results[(method_name, name)]['Train F1-Score (Weighted Avg)']:.2f}")
        print(f"\nTest Set Evaluation for {name} using {method_name} with SMOTE:")
        print(test_report)
        print(f"Confusion Matrix for {name} (Testing) using {method_name} with SMOTE:")
        print(test_confusion_matrix)
        print("="*60)



Using Feature Selection Method: ANOVA (f_classif)

Training KNN with ANOVA (f_classif) and SMOTE...

Weighted Average (Training) for KNN using ANOVA (f_classif) with SMOTE:
Precision: 0.54, Recall: 0.52, F1-Score: 0.48

Test Set Evaluation for KNN using ANOVA (f_classif) with SMOTE:
              precision    recall  f1-score   support

           0       0.20      0.06      0.10        16
           1       0.34      0.57      0.43        23
           2       0.75      0.69      0.72        55

    accuracy                           0.55        94
   macro avg       0.43      0.44      0.41        94
weighted avg       0.55      0.55      0.54        94

Confusion Matrix for KNN (Testing) using ANOVA (f_classif) with SMOTE:
[[ 1 11  4]
 [ 1 13  9]
 [ 3 14 38]]

Training Naive Bayes with ANOVA (f_classif) and SMOTE...

Weighted Average (Training) for Naive Bayes using ANOVA (f_classif) with SMOTE:
Precision: 0.64, Recall: 0.59, F1-Score: 0.53

Test Set Evaluation for Naive Bayes usin

smote with 0.5

In [17]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif, chi2, RFE, SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.preprocessing import PolynomialFeatures
from imblearn.over_sampling import SMOTE

# Define classifiers
classifiers = {
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "XGB": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "Extra Trees": ExtraTreesClassifier(),
    'SVM': SVC(random_state=42),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures()), ('model', LogisticRegression(max_iter=1000))])
}

# Define feature selection methods
feature_selection_methods = {
    "ANOVA (f_classif)": SelectKBest(f_classif, k=6),  # Adjust k as needed
    "Forward Selection": SequentialFeatureSelector(LogisticRegression(max_iter=1000), n_features_to_select=6, direction='forward', cv=5),
    "Backward Selection": SequentialFeatureSelector(LogisticRegression(max_iter=1000), n_features_to_select=6, direction='backward', cv=5),
    "Recursive Feature Elimination (RFE)": RFE(estimator=RandomForestClassifier(), n_features_to_select=6)  # Adjust n_features_to_select
}

# Initialize results storage
results = {}

# Set up cross-validation
kf = StratifiedKFold(n_splits=10)

# Assuming `X_scaled` and `y` are already prepared
# Replace these with your actual data
# Example: X_scaled = StandardScaler().fit_transform(X), y = target
# X_scaled and y should already exist at this point.

for method_name, selector in feature_selection_methods.items():
    print(f"\nUsing Feature Selection Method: {method_name}")

    # Perform feature selection
    if method_name in ["Forward Selection", "Backward Selection"]:
        selector.fit(X_scaled, y)
        X_selected = selector.transform(X_scaled)
    else:
        X_selected = selector.fit_transform(X_scaled, y)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, stratify=y, random_state=42)
    X_train_reset = pd.DataFrame(X_train).reset_index(drop=True)
    y_train_reset = pd.Series(y_train).reset_index(drop=True)

    # Calculate sampling strategy
    class_counts = y_train_reset.value_counts()
    majority_class_count = class_counts.max()
    sampling_strategy = {
        class_label: max(int(majority_class_count * 0.5), count) for class_label, count in class_counts.items()
    }

    # Initialize and apply SMOTE
    smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42)
    X_train_sm, y_train_sm = smote.fit_resample(X_train_reset, y_train_reset)

    # Train and evaluate classifiers
    for name, model in classifiers.items():
        print(f"\nTraining {name} with {method_name} and SMOTE...")

        # Cross-validation metrics
        cv_precision = []
        cv_recall = []
        cv_f1 = []

        # Perform Stratified K-Fold cross-validation
        for train_index, val_index in kf.split(X_train_sm, y_train_sm):
            X_fold_train, X_fold_val = X_train_sm.iloc[train_index], X_train_sm.iloc[val_index]
            y_fold_train, y_fold_val = y_train_sm.iloc[train_index], y_train_sm.iloc[val_index]

            # Fit model
            model.fit(X_fold_train, y_fold_train)

            # Predict and evaluate
            y_val_pred = model.predict(X_fold_val)
            report = classification_report(y_fold_val, y_val_pred, output_dict=True)

            # Collect weighted average metrics
            cv_precision.append(report['weighted avg']['precision'])
            cv_recall.append(report['weighted avg']['recall'])
            cv_f1.append(report['weighted avg']['f1-score'])

        # Compute average cross-validation metrics
        results[(method_name, name)] = {
            "Train Precision (Weighted Avg)": np.mean(cv_precision),
            "Train Recall (Weighted Avg)": np.mean(cv_recall),
            "Train F1-Score (Weighted Avg)": np.mean(cv_f1)
        }

        # Evaluate on the test set
        model.fit(X_train_sm, y_train_sm)
        y_test_pred = model.predict(X_test)
        test_report = classification_report(y_test, y_test_pred)
        test_confusion_matrix = confusion_matrix(y_test, y_test_pred)

        # Print results
        print(f"\nWeighted Average (Training) for {name} using {method_name} with SMOTE:")
        print(f"Precision: {results[(method_name, name)]['Train Precision (Weighted Avg)']:.2f}, "
              f"Recall: {results[(method_name, name)]['Train Recall (Weighted Avg)']:.2f}, "
              f"F1-Score: {results[(method_name, name)]['Train F1-Score (Weighted Avg)']:.2f}")
        print(f"\nTest Set Evaluation for {name} using {method_name} with SMOTE:")
        print(test_report)
        print(f"Confusion Matrix for {name} (Testing) using {method_name} with SMOTE:")
        print(test_confusion_matrix)
        print("=" * 60)



Using Feature Selection Method: ANOVA (f_classif)

Training KNN with ANOVA (f_classif) and SMOTE...

Weighted Average (Training) for KNN using ANOVA (f_classif) with SMOTE:
Precision: 0.55, Recall: 0.56, F1-Score: 0.52

Test Set Evaluation for KNN using ANOVA (f_classif) with SMOTE:
              precision    recall  f1-score   support

           0       0.20      0.06      0.10        16
           1       0.36      0.57      0.44        23
           2       0.75      0.73      0.74        55

    accuracy                           0.57        94
   macro avg       0.44      0.45      0.43        94
weighted avg       0.56      0.57      0.56        94

Confusion Matrix for KNN (Testing) using ANOVA (f_classif) with SMOTE:
[[ 1 11  4]
 [ 1 13  9]
 [ 3 12 40]]

Training Naive Bayes with ANOVA (f_classif) and SMOTE...

Weighted Average (Training) for Naive Bayes using ANOVA (f_classif) with SMOTE:
Precision: 0.61, Recall: 0.58, F1-Score: 0.54

Test Set Evaluation for Naive Bayes usin